In [4]:
import proteomics as ps

ImportError: cannot import name 'calculate_mol_mass' from 'proteomics.mass_spectra_simulation' (C:\Users\Carlo\Proteomics\proteomics\mass_spectra_simulation.py)

In [ ]:
protein_map = read_fasta("../data/sample_proteins.fasta")

protein_map

NameError: name 'ps' is not defined

In [1]:
import re

enzyme_cleavage_patterns = {
    'LysC': r'(?<=K)',
    'LysN': r'(?=K)',
    'ArgC': r'(?<=R)',
    'Trypsin': r'(?<=[KR])(?!P)',
}

enzyme_cleavage_patterns

{'LysC': '(?<=K)',
 'LysN': '(?=K)',
 'ArgC': '(?<=R)',
 'Trypsin': '(?<=[KR])(?!P)'}

In [ ]:
seq = protein_map["P11802"]
cleave_pattern = enzyme_cleavage_patterns["Trypsin"]
digest_protein_sequence(seq, cleave_pattern=cleave_pattern, min_pep_len=5, max_pep_len=30)

In [ ]:
cleave_pattern = enzyme_cleavage_patterns["Trypsin"]

digested_peptides_map = digest_protein_collection(
    protein_map,
    cleave_pattern= cleave_pattern,
    min_pep_len=5,
    max_pep_len=30,
)

digested_peptides_map

In [ ]:
prot_seq = protein_map["P11802"]
peps = digested_peptides_map["P11802"]

compute_sequence_coverage(prot_seq, peps)

In [ ]:
import pickle
output_path = '../results/digested_peptides_map.pkl'

with open(output_path, 'wb') as file:
    pickle.dump(digested_peptides_map, file)

In [ ]:
import pandas as pd

coverage_rows = []

for protease, cleavage_pattern in enzyme_cleavage_patterns.items():
    peptides_by_protein = digest_protein_collection(
        protein_map,
        cleave_pattern=cleavage_pattern,
        min_pep_len=5,
        max_pep_len=30,
    )

    for protein_id, protein_seq in protein_map.items():
        coverage_pct = compute_sequence_coverage(
            protein_seq,
            peptides_by_protein[protein_id],
        )

        coverage_rows.append({
            "protease": protease,
            "protein_id": protein_id,
            "coverage_pct": coverage_pct,
        })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df

In [ ]:
coverage_df.boxplot(column='coverage_pct', by='protease')

In [ ]:
import pickle

pickle_path = "../results/digested_peptides_map.pkl"  # e.g., '../results/digested_peptides_map.pkl'
with open(pickle_path, 'rb') as handle:
    digested_peptides_map = pickle.load(handle)

len(digested_peptides_map)

In [ ]:
pooled_peptides = []
for peptides in digested_peptides_map.values():
    pooled_peptides.extend(peptides) 
print(f'Total peptides: {len(pooled_peptides)}')

In [ ]:
rt_map = predict_lc_retention_times(pooled_peptides)

rt_map

In [ ]:
plot_retention_time(rt_map, resolution=30)

In [ ]:
select_peps = select_retention_time_window(
    peptide_rt_map=rt_map,
    lower_ret_time=0,
    upper_ret_time=20,
)

len(select_peps)

In [ ]:
import pickle
output_path = '../results/select_peptides.pkl'
with open(output_path, 'wb') as file:
    pickle.dump(select_peps, file)

In [ ]:
import pickle

with open('../results/select_peptides.pkl', 'rb') as handle:
    select_peps = pickle.load(handle)

len(select_peps)

In [ ]:
mz1_map = calculate_mz_collection(mass_map_ms1, charge=2)
mz1_map

In [ ]:
plot_spectrum(mz1_map, random_count_range=(0, 30000), seed=42)